# 12. Qualitative Translation Error & Synonym Analysis
Categorizes translation errors on example predictions -- run against real model output once available (notebook 09); the categorization logic itself is demonstrated here on illustrative examples.

In [ ]:
# ============================================================
# PATH & REPO AUTO-SYNC BOOSTER — Guarantees latest project code
# ============================================================
import os, sys, site, urllib.request, zipfile

# Purge cached 'src' modules from memory so updated files take effect immediately
for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

home       = os.path.expanduser('~')
proj_dir   = os.path.join(home, 'Ekegusii-LLM-Translation-main')
sync_tag   = os.path.join(proj_dir, 'configs', 'models', 'v2_mistral_earlystop_v5.tag')

# Auto-sync if folder is missing OR outdated (lacks v2_mistral_earlystop_v5.tag)
if not os.path.isfile(sync_tag):
    print('🔄 Outdated or missing repository detected. Auto-syncing latest code from GitHub...')
    zip_path = os.path.join(home, 'repo.zip')
    urllib.request.urlretrieve('https://github.com/aykahsay/Ekegusii-LLM-Translation/archive/refs/heads/main.zip', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(home)
    os.remove(zip_path)
    print('✅ Repository auto-synced to latest main commit!')

if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')


In [2]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


numpy/pandas ABI OK.


In [3]:
import pandas as pd

# Illustrative (source, reference, hypothesis) triples -- replace with
# real predictions saved from notebook 09 for genuine error analysis.
examples = pd.DataFrame({
    'source': [
        'Wash your hands frequently with soap.',
        'The Ministry of Health announced a new vaccination campaign.',
        'Farmers should plant drought-resistant crops.',
    ],
    'reference': [
        'Esibie amaboko ao botambe na esabuni.',
        'Ewizara ya obochenu yatangaza omochenu mocha ogotema.',
        'Abasaki bagoika gotema amakoro agatangete oborwa amanche.',
    ],
    'hypothesis': [
        'Esibie amaboko ao botambe na esabuni.',
        'Ewizara ya obochenu yatangaza omochenu.',
        'Abasaki bagoika gotema.',
    ],
})
examples

,source,reference,hypothesis
0,Wash your hands frequently with soap.,Esibie amaboko ao botambe na esabuni.,Esibie amaboko ao botambe na esabuni.
1,The Ministry of Health announced a new vaccina...,Ewizara ya obochenu yatangaza omochenu mocha o...,Ewizara ya obochenu yatangaza omochenu.
2,Farmers should plant drought-resistant crops.,Abasaki bagoika gotema amakoro agatangete obor...,Abasaki bagoika gotema.


## Length-ratio error flag (a cheap proxy for omission/truncation)

In [4]:
def word_count(text):
    return len(str(text).split())

examples['ref_len'] = examples['reference'].apply(word_count)
examples['hyp_len'] = examples['hypothesis'].apply(word_count)
examples['length_ratio'] = examples['hyp_len'] / examples['ref_len']
examples['likely_omission'] = examples['length_ratio'] < 0.7
examples[['source', 'ref_len', 'hyp_len', 'length_ratio', 'likely_omission']]

,source,ref_len,hyp_len,length_ratio,likely_omission
0,Wash your hands frequently with soap.,6,6,1.000000,False
1,The Ministry of Health announced a new vaccina...,7,5,0.714286,False
2,Farmers should plant drought-resistant crops.,7,3,0.428571,True


## Exact-match flag

In [5]:
examples['exact_match'] = examples['reference'].str.strip() == examples['hypothesis'].str.strip()
examples[['source', 'exact_match', 'likely_omission']]

,source,exact_match,likely_omission
0,Wash your hands frequently with soap.,True,False
1,The Ministry of Health announced a new vaccina...,False,False
2,Farmers should plant drought-resistant crops.,False,True


## Error category summary
In a real run, extend this with: rare-word-containing sentences (`RareWordAccuracyEvaluator.split_by_rarity`), terminology mismatches (`TerminologyConsistencyChecker.check_translation`), and per-domain breakdowns joined from the source corpus's `Domain`/`source` column.

In [6]:
summary = {
    'total_examples': len(examples),
    'exact_matches': int(examples['exact_match'].sum()),
    'likely_omissions': int(examples['likely_omission'].sum()),
}
summary

{'total_examples': 3, 'exact_matches': 1, 'likely_omissions': 1}